## base import and API modules

In [38]:
import requests
import json

from contextlib import redirect_stdout
import io

# Global variables for pagination, easy to tweak here
DEFAULT_PAGE = 0
DEFAULT_SIZE = 25
BASE_URL = "http://localhost:8080/api"

def print_response(response):
    """Utility to print responses clearly (indented JSON or plain string)"""
    print(f"Status Code: {response.status_code}")
    try:
        # Try to parse and print nicely formatted JSON
        parsed_json = response.json()
        print(json.dumps(parsed_json, indent=4))
    except ValueError:
        # If it fails (e.g., plain text like "The game has been added"), print raw text
        text = response.text
        if text:
            print(text)
        else:
            print("[No content in response body]")
    print("=" * 50 + "\n")

def print_json(target_json):
    print(json.dumps(target_json, indent=4))

def mute(func, *args, **kwargs):
    """Prevents output of function func"""
    with redirect_stdout(io.StringIO()):
        return func(*args, **kwargs)

def login_as(username):
    api.logout()
    my_payload = {
        "email" : username + "@hotmail.com",
        "password" : username
    }
    response = api.login(my_payload)
    if(response.status_code != 200):
        raise ValueError("ERROR DURING LOGIN")
    else:
        print("You are now logged as: " + username)

def get_user_id(username):
    src_res = api.search_user(username,0,1).json()["content"]
    if(len(src_res) != 1):
        raise ValueError("USER NOT FOUND")
    return src_res[0]["id"]

class PlayerHiveClient:
    def __init__(self, base_url):
        self.base_url = base_url
        self.session = requests.Session()
        self.session.headers.update({"Content-Type": "application/json"})
        self.token = None

    def set_token(self, token):
        """Sets the JWT token for authenticated requests"""
        self.token = token
        self.session.headers.update({"Authorization": f"Bearer {token}"})
        print("[+] Token set successfully. You are now authenticated.\n")

    def logout(self):
        """Clears the token to allow switching accounts or testing unauthenticated routes"""
        self.token = None
        if "Authorization" in self.session.headers:
            del self.session.headers["Authorization"]
        print("[-] Token cleared. You are now logged out.\n")

    def _request(self, method, endpoint, **kwargs):
        """Base method to execute and print the request"""
        url = f"{self.base_url}{endpoint}"
        print(f"--- {method} {url} ---")
        response = self.session.request(method, url, **kwargs)
        return response

#### API MODULES
class API(PlayerHiveClient):
    
    # ================= AUTH =================
    def register(self, payload={}):
        return self._request("POST", "/auth/register", json=payload)

    def login(self, payload={}):
        response = self._request("POST", "/auth/login", json=payload)
        if response.status_code == 200:
            token = response.json().get("accessToken")
            if token:
                self.set_token(token)
        return response

    # ================= USER =================
    def get_own_profile(self):
        return self._request("GET", "/user/MyProfile")

    def get_user_profile(self, user_id):
        return self._request("GET", f"/user/{user_id}")
    
    def get_own_library(self,page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/user/MyLibrary", params=params)

    def get_user_library(self, user_id, page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/user/library/{user_id}", params=params)

    def edit_library(self, payload={}):
        return self._request("POST", "/user/editLibrary", json=payload)

    def remove_from_library(self, game_id):
        return self._request("DELETE", f"/user/removeFromLibrary/{game_id}")

    def get_friend_list(self, user_id, page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/user/friends/{user_id}", params=params)

    def get_friend_requests(self,page=DEFAULT_PAGE,size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", "/user/friendRequests", params = params)

    def search_user(self, query, page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/user/search/{query}", params=params)

    def send_friend_request(self, target_user_id):
        return self._request("POST", f"/user/sendFriendRequest/{target_user_id}")

    def approve_friend_request(self, target_user_id):
        return self._request("POST", f"/user/approveFriendRequest/{target_user_id}")

    def deny_friend_request(self, target_user_id):
        return self._request("DELETE", f"/user/denyFriendRequest/{target_user_id}")

    def remove_friend(self, friend_id):
        return self._request("DELETE", f"/user/removeFriend/{friend_id}")
    
    def get_user_reviews(self, user_id, page=DEFAULT_PAGE,size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET","/user/reviews/" + user_id, params = params)
    
    def get_own_reviews(self, page=DEFAULT_PAGE,size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET","/user/MyReviews", params = params)

    def delete_account(self):
        return self._request("DELETE", "/user/deleteAccount")

    # ================= GAMES =================
    def get_game_info(self, game_id):
        return self._request("GET", f"/games/{game_id}")

    def search_game(self, game_name, page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/games/search/{game_name}", params=params)

    def get_game_reviews(self, game_id, page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/games/showReviews/{game_id}", params=params)

    def add_review(self, game_id, payload={}):
        return self._request("POST", f"/games/addReview/{game_id}", json=payload)

    def delete_review(self, review_id):
        return self._request("DELETE", f"/games/deleteReview/{review_id}")

    # ================= ADMIN =================
    def add_game(self, payload={}):
        return self._request("POST", "/admin/addGame", json=payload)

    def edit_game(self, game_id, payload={}):
        return self._request("PATCH", f"/admin/editGame/{game_id}", json=payload)

    def delete_game(self, game_id):
        return self._request("DELETE", f"/admin/deleteGame/{game_id}")

    def delete_user_admin(self, user_id):
        return self._request("DELETE", f"/admin/deleteUser/{user_id}")

# Initialize the API object
api = API(BASE_URL)


## LOGIN SECTION

### Register user

In [ ]:
print(">>> REGISTER AS STANDARD USER")
register_username = "fabriziomaggi"
user_payload = {
    "username" : register_username,
    "email" : register_username + "@hotmail.com",
    "password" : "sederone",
    "birthDate" : "2002-08-29"

}
last_register_info = {
    "email": user_payload["email"],
    "password" : user_payload["password"]
}
print_response(api.register(user_payload))

### Login last registered user

In [ ]:
print(">>> LOGIN AS LAST REGISTERED USER: " + register_username)

print_response(api.login(last_register_info))

### General Login

In [ ]:
print(">>> LOGIN AS STANDARD USER")
user_payload = {
    "email" : "fabriziomaggi@hotmail.com",
    "password" : "sederone"

}
print_response(api.login(user_payload))

### Logout

In [ ]:
print(">>> DROPPING TOKEN")
api.logout()

## USER OPERATIONS

### Profile API tests

In [ ]:
print(">>> LOGGING OUT")
api.logout()

In [ ]:
print(">>> TEST 1: USER SEARCH")

user_search_target_string = "br"
user_search_result = api.search_user(user_search_target_string,0,10)

print_response(user_search_result)
# print_json(user_search_result.json()["content"])

In [ ]:
print(">>> TEST 2: PROFILE REQUEST")
print(">>>>> asking for "+ user_search_result.json()["content"][0]["username"] + "'s profile") # the first user obtained by the previous query

print_response(api.get_user_profile(user_search_result.json()["content"][0]["id"]))

In [ ]:
print(">>> TEST 3: PROFILE REQUEST WITH LOGIN")

api.logout()
api.login(last_register_info)

print_response(api.get_user_profile(user_search_result.json()["content"][0]["id"]))

In [ ]:
print(">>> TEST 4: OWN PROFILE REQUEST, BOTH WITH ID AND WITHOUT")

api.logout()
api.login(last_register_info)

last_register_id = api.search_user(register_username,0,1).json()["content"][0]["id"]

my_profile_without_id = api.get_user_profile(last_register_id) # get the id of my account

my_profile_with_id = api.get_own_profile()

print_response(my_profile_with_id)
if(my_profile_without_id.json() == my_profile_with_id.json()):
    print("=========== The two queries are equal ==========")
else:
    print("======= ERROR! WITHOUT ID:")
    print_response(my_profile_without_id)

In [ ]:
print(">>> TEST 5: LIBRARY REQUEST")
target_user = get_user_id("kristinmullen")

page_number = 0
page_size = 11

print_response(api.get_user_library(target_user, page_number, page_size))

In [ ]:
print(">>> TEST 6: OWN LIBRARY REQUEST")
login_as("fabriziomaggi")

page_number = 0
page_size = 2

print_response(api.get_own_library( page_number, page_size))

In [ ]:
print(">>> TEST ?: PAGED FRIEND REQUESTS")

login_as("torresmary")

page_number = 0
page_size = 2

print_response(api.get_friend_requests(page_number,page_size))

>>> TEST ?: PAGED FRIEND REQUESTS
[-] Token cleared. You are now logged out.

--- GET http://localhost:8080/api/user/friendRequests ---
Status Code: 403
[No content in response body]



In [50]:
print(">>> TEST ?: PAGED USER REVIEWS")

page_number =0
page_size = 8


print_response(api.get_user_reviews("1f4fb08bcae24262bad17f57",page_number,page_size))

>>> TEST ?: PAGED USER REVIEWS
--- GET http://localhost:8080/api/user/reviews/1f4fb08bcae24262bad17f57 ---
Status Code: 200
[
    {
        "id": "5b902a6eac7b4395bbf23e77",
        "userId": "1f4fb08bcae24262bad17f57",
        "username": "torresmary",
        "pfpURL": "/Playerhive/pfp/0fff586e37b54f05b9e43e237a1434dd",
        "reviewText": "Score: 9/10  Plot: 15 years after the events of the 1979 Alien movie, Alien: Isolation picks up. You play as Amanda Ripley as she finds out there may have been a break on the case of her mother's disappeance. Amanda heads out to Sevastopol Station to recover the black-box of her mother's ship and immediately things go horribly wrong. The station's androids are going haywire and soon Amanda meets the Alien responsible for the mass chaos aboard the station. Amanda has to sneak around the station avoiding the Alien, androids, and the hostile station crew.   Positives: Very interesting story that is very well written. The game is very suspenseful, a